In [1]:
# ================================================================
# CELL 1 — Imports
# ================================================================
import os, re, json, warnings, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from transformers import (
    AutoModel, AutoTokenizer, AutoModelForSequenceClassification,
    RobertaModel, RobertaTokenizer
)
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
torch._dynamo.config.suppress_errors = True
os.environ['TORCHDYNAMO_DISABLE'] = '1'

# ── Paths ─────────────────────────────────────────────────────
SAVE_DIR     = './models_v3'
GNN_DIR      = './mode'
EWC_DIR      = './ewc/ewc'   # roberta_ewc_best.pt | bart_ewc_best.pt | llama_ewc_best.pt
DATASET_PATH = './This_final.csv'
FAISS_DIR    = './mode'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')


✅ Device: cuda


In [2]:
# ================================================================
# CELL 2 — Dataset + SBERT + Node features
# ================================================================
df = pd.read_csv(DATASET_PATH)
df['label']         = df['label'].astype(int)
df['content_clean'] = df['content_clean'].fillna('').str.strip()
df['source']        = df['source'].fillna('unknown').astype(str)
df['date']          = pd.to_datetime(df['date'], errors='coerce')
df['date_confidence'] = df['date_confidence'].fillna(0.2)
min_date = df['date'].min(); max_date = df['date'].max()
df['time_norm'] = (
    (df['date'] - min_date).dt.total_seconds() /
    (max_date - min_date).total_seconds()
).fillna(0.5)
df = df[df['content_clean'] != ''].reset_index(drop=True)
print(f'Dataset : {len(df):,} articles')

sbert = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=str(device))
sbert_dim = sbert.get_sentence_embedding_dimension()
print(f'✅ SBERT chargé — dim={sbert_dim}')

X_graph     = torch.load(f'{GNN_DIR}/node_features_sbert2.pt')
IN_CHANNELS = X_graph.shape[1]
print(f'Node features : {X_graph.shape}')


Dataset : 17,463 articles


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ SBERT chargé — dim=768
Node features : torch.Size([17463, 770])


In [3]:
# ================================================================
# CELL 3 — Définitions des architectures
# ================================================================

class FakeNewsClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        hidden_size  = self.roberta.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )
    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(out.last_hidden_state[:, 0, :])

class BARTFakeNewsClassifier(nn.Module):
    def __init__(self, model_name='facebook/bart-base', unfreeze_last=1, dropout=0.4):
        super().__init__()
        self.bart = AutoModel.from_pretrained(model_name)
        hidden_size = self.bart.config.d_model
        for param in self.bart.parameters():
            param.requires_grad = False
        for p in self.bart.encoder.layers[-unfreeze_last:]:
            for w in p.parameters(): w.requires_grad = True
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64),          nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    def forward(self, input_ids, attention_mask):
        out  = self.bart(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.classifier(pooled)

class GCNLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.linear = nn.Linear(in_ch, out_ch)
    def forward(self, x, adj):
        return self.linear(torch.sparse.mm(adj, x))

class SourceGCN(nn.Module):
    def __init__(self, in_ch, hidden, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_ch, hidden)
        self.conv2 = GCNLayer(hidden, hidden)
        self.conv3 = GCNLayer(hidden, hidden // 2)
        self.conv4 = GCNLayer(hidden // 2, num_classes)
        self.ln1   = nn.LayerNorm(hidden)
        self.ln2   = nn.LayerNorm(hidden)
        self.ln3   = nn.LayerNorm(hidden // 2)
        self.proj  = nn.Linear(in_ch, hidden)
    def forward(self, x, adj):
        h1 = F.relu(self.ln1(self.conv1(x, adj))); h1 = h1 + self.proj(x)
        h2 = F.relu(self.ln2(self.conv2(h1, adj))); h2 = h2 + h1
        h3 = F.relu(self.ln3(self.conv3(h2, adj)))
        return self.conv4(h3, adj)

class RelationalGAT(nn.Module):
    def __init__(self, in_ch, hidden, num_classes):
        super().__init__()
        self.W_a2s    = nn.Linear(in_ch, hidden)
        self.bn_src   = nn.BatchNorm1d(hidden)
        self.W_s2a    = nn.Linear(hidden, hidden)
        self.bn_art   = nn.BatchNorm1d(hidden)
        self.W_a2s2   = nn.Linear(hidden, hidden)
        self.bn_src2  = nn.BatchNorm1d(hidden)
        self.W_s2a2   = nn.Linear(hidden, hidden)
        self.bn_art2  = nn.BatchNorm1d(hidden)
        self.residual = nn.Linear(in_ch, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.5),
            nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(0.5),     nn.Linear(64, num_classes)
        )
        self.dropout = nn.Dropout(0.5)
    def forward(self, x_art, x_src, A_a2s, A_s2a):
        h_src  = F.relu(self.bn_src(self.W_a2s(torch.sparse.mm(A_a2s, x_art))))
        h_src  = self.dropout(h_src)
        h_art  = F.relu(self.bn_art(self.W_s2a(torch.sparse.mm(A_s2a, h_src))))
        h_art  = h_art + self.residual(x_art); h_art = self.dropout(h_art)
        h_src2 = F.relu(self.bn_src2(self.W_a2s2(torch.sparse.mm(A_a2s, h_art))))
        h_src2 = self.dropout(h_src2)
        h_art2 = F.relu(self.bn_art2(self.W_s2a2(torch.sparse.mm(A_s2a, h_src2))))
        h_art2 = h_art2 + h_art; h_art2 = self.dropout(h_art2)
        return self.classifier(h_art2)

class GraphSAGELayer(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.4):
        super().__init__()
        self.linear  = nn.Linear(in_ch * 2, out_ch)
        self.ln      = nn.LayerNorm(out_ch)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, adj):
        agg = torch.sparse.mm(adj, x)
        out = self.linear(torch.cat([x, agg], dim=1))
        return F.relu(self.ln(self.dropout(out)))

class KnowledgeGNN(nn.Module):
    def __init__(self, in_ch, hidden, num_classes):
        super().__init__()
        self.input_dropout = nn.Dropout(0.3)
        self.sage1    = GraphSAGELayer(in_ch, hidden, 0.4)
        self.sage2    = GraphSAGELayer(hidden, hidden // 2, 0.4)
        self.residual = nn.Linear(in_ch, hidden // 2)
        self.dropout  = nn.Dropout(0.4)
        self.classifier = nn.Sequential(
            nn.Linear(hidden // 2, 64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64, num_classes)
        )
    def forward(self, x, adj):
        x   = self.input_dropout(x)
        res = self.residual(x)
        x   = self.sage2(self.sage1(x, adj), adj) + res
        return self.classifier(self.dropout(x))

print('✅ Architectures définies')


✅ Architectures définies


In [4]:
# ================================================================
# CELL 4 — Chargement des modèles (Transformers EWC + GNNs)
# ================================================================
import gc
gc.collect(); torch.cuda.empty_cache()

# ── RoBERTa EWC ───────────────────────────────────────────────
rob_tokenizer     = RobertaTokenizer.from_pretrained('roberta-base')
roberta_ewc_model = FakeNewsClassifier().to(device)
roberta_ewc_model.load_state_dict(
    torch.load(f'{EWC_DIR}/roberta_ewc_best.pt',
               map_location=device, weights_only=False))
roberta_ewc_model.eval()
print('✅ RoBERTa EWC chargé')

# ── BART EWC ──────────────────────────────────────────────────
bart_tokenizer = AutoTokenizer.from_pretrained('facebook/bart-base')
bart_ewc_model = BARTFakeNewsClassifier().to(device)
bart_ewc_model.load_state_dict(
    torch.load(f'{EWC_DIR}/bart_ewc_best.pt',
               map_location=device, weights_only=False))
bart_ewc_model.eval()
print('✅ BART EWC chargé')

# ── LLaMA EWC ─────────────────────────────────────────────────
gc.collect(); torch.cuda.empty_cache()

llama_model_name = 'meta-llama/Llama-3.2-1B'
llama_tokenizer  = AutoTokenizer.from_pretrained(llama_model_name)
llama_tokenizer.pad_token    = llama_tokenizer.eos_token
llama_tokenizer.padding_side = 'right'

llama_ewc_model = AutoModelForSequenceClassification.from_pretrained(
    llama_model_name, num_labels=2,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True)   # ← pas de device_map
llama_ewc_model.config.pad_token_id = llama_tokenizer.pad_token_id

state = torch.load(f'{EWC_DIR}/llama_ewc_best.pt', map_location='cpu', weights_only=False)
llama_ewc_model.load_state_dict(state, strict=False)
del state; gc.collect()

llama_ewc_model = llama_ewc_model.to(device)
torch.cuda.empty_cache()
llama_ewc_model.eval()
print('✅ LLaMA EWC chargé')

# ── GNNs ──────────────────────────────────────────────────────
gnn1_model = SourceGCN(IN_CHANNELS, 256, 2).to(device)
gnn1_model.load_state_dict(
    torch.load(f'{GNN_DIR}/best_gnn1_sbert.pt', map_location=device, weights_only=False))
gnn1_model.eval()

gnn2_model = RelationalGAT(IN_CHANNELS, 256, 2).to(device)
gnn2_model.load_state_dict(
    torch.load(f'{GNN_DIR}/best_gnn2_sbert2.pt', map_location=device, weights_only=False))
gnn2_model.eval()

gnn3_model = KnowledgeGNN(IN_CHANNELS, 128, 2).to(device)
gnn3_model.load_state_dict(
    torch.load(f'{GNN_DIR}/best_gnn3_sbert2.pt', map_location=device, weights_only=False))
gnn3_model.eval()
print('✅ GNN1 / GNN2 / GNN3 chargés')

# ── Meta-learner ──────────────────────────────────────────────
trans_preds = np.load(f'{SAVE_DIR}/hard_voting_final.npy')
gnn_preds   = np.load(f'{GNN_DIR}/hard_voting_gnns.npy')
rag_preds   = np.load(f'{GNN_DIR}/rag_preds_test.npy')
true_labels = np.load(f'{SAVE_DIR}/hard_voting_trans_labs.npy')
N = min(len(trans_preds), len(gnn_preds), len(rag_preds), len(true_labels))
X_meta = np.column_stack([trans_preds[:N], gnn_preds[:N], rag_preds[:N]]).astype(float)
y_true = true_labels[:N]
meta_lr = LogisticRegression(class_weight={0:1.0,1:2.0}, random_state=42, max_iter=1000)
meta_lr.fit(X_meta, y_true)
print('✅ Meta-learner (LogReg) entraîné')

# ── SHAP sur stacking ─────────────────────────────────────────
explainer_lr     = shap.LinearExplainer(meta_lr, X_meta, feature_perturbation='interventional')
shap_vals_global = np.abs(explainer_lr.shap_values(X_meta))
if isinstance(shap_vals_global, list): shap_vals_global = shap_vals_global[1]
feat_importance  = shap_vals_global.mean(0)
print(f'SHAP → Trans:{feat_importance[0]:.4f} | GNN:{feat_importance[1]:.4f} | RAG:{feat_importance[2]:.4f}')
print('\n✅ Tous les modèles chargés (Transformers EWC + GNNs originaux)')


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ RoBERTa EWC chargé


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

✅ BART EWC chargé


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ LLaMA EWC chargé
✅ GNN1 / GNN2 / GNN3 chargés
✅ Meta-learner (LogReg) entraîné
SHAP → Trans:1.9845 | GNN:1.5164 | RAG:0.8029

✅ Tous les modèles chargés (Transformers EWC + GNNs originaux)


In [5]:
# ================================================================
# CELL 5 — Chargement FAISS + KB RAG
# ================================================================
import faiss
sbert_rag = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=str(device))

FAISS_CANDIDATES = [
    (f"{FAISS_DIR}/kb_faiss_v2.index", f"{FAISS_DIR}/kb_docs_v2.json"),
    (f"{SAVE_DIR}/kb_faiss_v2.index",  f"{SAVE_DIR}/kb_docs_v2.json"),
]

faiss_index, kb_docs, kb_meta = None, [], []
for idx_path, docs_path in FAISS_CANDIDATES:
    if os.path.exists(idx_path) and os.path.exists(docs_path):
        faiss_index = faiss.read_index(idx_path)
        with open(docs_path, encoding='utf-8') as f:
            saved = json.load(f)
        kb_docs = saved['documents']
        kb_meta = saved['metadata']
        print(f'✅ FAISS chargé : {faiss_index.ntotal} vecteurs | KB : {len(kb_docs)} docs')
        break
if faiss_index is None:
    print('⚠️  FAISS introuvable — RAG utilisera BART EWC uniquement')

SOURCE_CREDIBILITY = {
    'politifact': 0.97, 'guardian': 0.92, 'web.archive.org': 0.85,
    'isot': 0.75, 'serpapi': 0.72, 'new_articles_2025': 0.72,
    'train_real': 0.70, 'train_fake': 0.70, 'default': 0.60,
}
def get_credibility(meta):
    src = str(meta.get('source', '')).lower()
    for k, v in SOURCE_CREDIBILITY.items():
        if k in src: return v
    return SOURCE_CREDIBILITY['default']

# RAG utilise BART EWC
BART_RAG_TOK = bart_tokenizer
BART_RAG_MOD = bart_ewc_model
print('✅ RAG configuré avec BART EWC')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚠️  FAISS introuvable — RAG utilisera BART EWC uniquement
✅ RAG configuré avec BART EWC


In [6]:
# ================================================================
# CELL 6 — Fonctions d'inférence (Transformers EWC)
# ================================================================

def predict_roberta(text: str) -> int:
    """RoBERTa EWC — adapté aux articles récents multi-domaines"""
    tokens = rob_tokenizer(
        ' '.join(text.split()[:512]), return_tensors='pt',
        truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        logits = roberta_ewc_model(tokens['input_ids'], tokens['attention_mask'])
    return int(logits.argmax(dim=-1).item())

def predict_roberta_proba(text: str):
    """RoBERTa EWC — retourne probabilités [P(REAL), P(FAKE)]"""
    tokens = rob_tokenizer(
        ' '.join(text.split()[:512]), return_tensors='pt',
        truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        logits = roberta_ewc_model(tokens['input_ids'], tokens['attention_mask'])
        prob   = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return prob

def predict_bart(text: str) -> int:
    """BART EWC — adapté aux articles récents multi-domaines"""
    tokens = bart_tokenizer(
        ' '.join(text.split()[:512]), return_tensors='pt',
        truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        logits = bart_ewc_model(tokens['input_ids'], tokens['attention_mask'])
    return int(logits.argmax(dim=-1).item())

def predict_llama(text: str) -> int:
    """LLaMA EWC — adapté aux articles récents multi-domaines"""
    tokens = llama_tokenizer(
        ' '.join(text.split()[:512]), return_tensors='pt',
        truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = llama_ewc_model(
            input_ids=tokens['input_ids'],
            attention_mask=tokens['attention_mask'])
    return int(out.logits.argmax(dim=-1).item())

def normalize_adj_sparse(edge_src, edge_dst, n, edge_weights=None):
    if edge_weights is None:
        edge_weights = np.ones(len(edge_src), dtype=np.float32)
    A = sp.coo_matrix(
        (edge_weights.astype(np.float32), (edge_src.astype(np.int32), edge_dst.astype(np.int32))),
        shape=(n, n))
    A = A + A.T + sp.eye(n, dtype=np.float32)
    deg = np.array(A.sum(axis=1)).flatten()
    d_inv = np.power(deg, -0.5, where=deg > 0)
    d_inv[np.isinf(d_inv)] = 0.0
    D = sp.diags(d_inv)
    A_norm = (D @ A @ D).tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack([A_norm.row, A_norm.col])).long()
    values  = torch.from_numpy(A_norm.data)
    return torch.sparse_coo_tensor(indices, values, torch.Size(A_norm.shape)).to(device)

def predict_gnns(text: str, source: str = 'unknown', time_norm: float = 0.5):
    text_trunc = ' '.join(text.split()[:512])
    new_emb    = sbert.encode([text_trunc], normalize_embeddings=True,
                               convert_to_tensor=False).astype('float32')
    extra      = np.array([[time_norm, 0.5]], dtype=np.float32)
    new_feat   = np.concatenate([new_emb, extra], axis=1)
    new_feat_t = torch.tensor(new_feat)
    N = len(X_graph); new_idx = N
    X_ext = torch.cat([X_graph, new_feat_t], dim=0)
    K = 5
    existing_embs = X_graph[:, :768].numpy()
    nbrs = NearestNeighbors(n_neighbors=K, metric='cosine', n_jobs=-1)
    nbrs.fit(existing_embs)
    dists, idxs = nbrs.kneighbors(new_emb)
    edge_src_new = np.array([new_idx] * K + list(idxs[0]))
    edge_dst_new = np.array(list(idxs[0]) + [new_idx] * K)
    edge_w_new   = np.array([1 - d for d in dists[0]] * 2, dtype=np.float32)
    es = np.load('./mode/edge_src_cosine2.npy')
    ed = np.load('./mode/edge_dst_cosine2.npy')
    ew = np.load('./mode/edge_weights_cosine2.npy')
    edge_src_all = np.concatenate([es, edge_src_new])
    edge_dst_all = np.concatenate([ed, edge_dst_new])
    edge_w_all   = np.concatenate([ew, edge_w_new])
    adj   = normalize_adj_sparse(edge_src_all, edge_dst_all, N + 1, edge_w_all)
    X_dev = X_ext.to(device)
    preds = []
    with torch.no_grad():
        out1 = gnn1_model(X_dev, adj)
        pred1 = int(out1[new_idx].argmax().item())
        preds.append(pred1)
    with torch.no_grad():
        out3 = gnn3_model(X_dev, adj)
        pred3 = int(out3[new_idx].argmax().item())
        preds.append(pred3)
    unique_sources = list(df['source'].unique()) + [source]
    source2idx     = {s: i for i, s in enumerate(unique_sources)}
    num_sources    = len(unique_sources)
    source_feats   = torch.zeros(num_sources, IN_CHANNELS)
    source_count   = torch.zeros(num_sources)
    for i, s in enumerate(df['source']):
        sidx = source2idx[s]
        source_feats[sidx] += X_graph[i]; source_count[sidx] += 1
    new_sidx = source2idx[source]
    source_feats[new_sidx] += new_feat_t[0]; source_count[new_sidx] += 1
    source_feats = source_feats / source_count.unsqueeze(1).clamp(min=1)
    art_indices = list(range(N + 1))
    src_indices = [source2idx[s] for s in list(df['source']) + [source]]
    conf_w      = list(df['date_confidence'].values) + [0.5]
    A_a2s = torch.sparse_coo_tensor(
        torch.tensor([src_indices, art_indices], dtype=torch.long),
        torch.tensor(conf_w, dtype=torch.float), (num_sources, N + 1)).to(device)
    A_s2a = torch.sparse_coo_tensor(
        torch.tensor([art_indices, src_indices], dtype=torch.long),
        torch.ones(N + 1), (N + 1, num_sources)).to(device)
    with torch.no_grad():
        out2 = gnn2_model(X_dev, source_feats.to(device), A_a2s, A_s2a)
        pred2 = int(out2[new_idx].argmax().item())
        preds.append(pred2)
    gnn_vote = int(sum(preds) >= 2)
    return gnn_vote, pred1, pred2, pred3

def extract_top3_claim(text: str) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if len(s.split()) >= 8]
    if len(sentences) <= 3: return ' '.join(sentences)
    embs = sbert_rag.encode(sentences, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False)
    centroid = embs.mean(0); centroid /= np.linalg.norm(centroid) + 1e-9
    top = sorted(np.argsort(embs @ centroid)[::-1][:3])
    return ' '.join(sentences[i] for i in top)

def predict_rag(text: str, conf_threshold: float = 0.40):
    claim = extract_top3_claim(text)
    retrieved_docs = []
    def retrieve(query):
        if faiss_index is None: return [], 0.0
        q = sbert_rag.encode([query], normalize_embeddings=True,
                              convert_to_numpy=True).astype('float32')
        scores, idxs = faiss_index.search(q, 20)
        docs, seen = [], {}
        for sim, idx in zip(scores[0], idxs[0]):
            if len(docs) >= 5 or idx < 0: continue
            meta = kb_meta[idx]; src = meta.get('source', 'unknown')
            seen[src] = seen.get(src, 0) + 1
            if seen[src] > 2: continue
            sim_f = max(0.0, float(sim))
            docs.append({'label_int': int(meta.get('label_int', 0)),
                         'wscore': sim_f * get_credibility(meta),
                         'text': kb_docs[idx][:120],
                         'source': src})
        conf = max((d['wscore'] for d in docs), default=0.0)
        return docs, conf
    docs_r1, conf_r1 = retrieve(claim)
    if conf_r1 >= conf_threshold:
        best_docs = docs_r1
    else:
        entities = re.findall(r'\b[A-Z][a-z]{2,}\b', text)
        query_r2  = ' '.join(list(dict.fromkeys(entities))[:10]) or text[:250]
        docs_r2, conf_r2 = retrieve(query_r2)
        best_docs = docs_r2 if conf_r2 >= conf_r1 + 0.05 else (docs_r1[:3] + docs_r2[:3])
    retrieved_docs = best_docs
    total_w = sum(d['wscore'] for d in best_docs)
    kb_real = sum(d['wscore'] for d in best_docs if d['label_int'] == 0) / max(total_w, 1e-8) if best_docs else 0.5
    tokens  = BART_RAG_TOK(claim[:512], return_tensors='pt',
                           truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        probs_bart = F.softmax(
            BART_RAG_MOD(tokens['input_ids'], tokens['attention_mask']), dim=-1
        ).cpu().numpy()[0]
    bart_real = float(probs_bart[0])
    rag_score = 0.65 * kb_real + 0.35 * bart_real
    rag_pred  = int(rag_score < 0.65)
    return rag_pred, retrieved_docs, float(1 - rag_score if rag_pred == 1 else rag_score)

def get_shap_tokens(text: str, n_top: int = 15):
    masker = shap.maskers.Text(rob_tokenizer)
    def predict_fn(texts):
        probs = []
        for t in texts:
            tokens = rob_tokenizer(str(t), return_tensors='pt',
                truncation=True, max_length=512, padding='max_length').to(device)
            with torch.no_grad():
                logits = roberta_ewc_model(tokens['input_ids'], tokens['attention_mask'])
                prob   = torch.softmax(logits, dim=1).cpu().numpy()[0]
            probs.append(prob)
        return np.array(probs)
    explainer = shap.Explainer(predict_fn, masker, output_names=['REAL', 'FAKE'])
    sv  = explainer([text[:400]], fixed_context=1)
    vals   = sv[0, :, 1].values
    toks   = sv[0, :, 1].data
    idx    = np.argsort(np.abs(vals))[-n_top:][::-1]
    return [str(toks[i]) for i in idx], vals[idx]

print('✅ Fonctions d\'inférence EWC prêtes (RoBERTa EWC + BART EWC + LLaMA EWC)')

# Test rapide
print('\n🔍 Test rapide...')
t1 = 'BREAKING: Aliens confirmed in Washington DC by NASA scientists.'
t2 = 'The Federal Reserve raised interest rates by 25 basis points on Wednesday.'
for t, expected in [(t1,'FAKE'), (t2,'REAL')]:
    r = predict_roberta(t); b = predict_bart(t); l = predict_llama(t)
    hv = 'FAKE' if (r+b+l) >= 2 else 'REAL'
    print(f'  [{expected}] Rob={"F" if r else "R"} Bart={"F" if b else "R"} LLaMA={"F" if l else "R"} → HV={hv} {"✅" if hv==expected else "❌"}')


✅ Fonctions d'inférence EWC prêtes (RoBERTa EWC + BART EWC + LLaMA EWC)

🔍 Test rapide...
  [FAKE] Rob=F Bart=F LLaMA=F → HV=FAKE ✅
  [REAL] Rob=R Bart=F LLaMA=F → HV=FAKE ❌


In [7]:
# ================================================================
# CELL 7 — Pipeline complet predict_full()
# ================================================================
import time

def predict_full(text: str, source: str = 'unknown') -> dict:
    t0     = time.perf_counter()
    timings = {}

    # ── Niveau 1 : Transformers EWC ───────────────────────────
    rob_prob   = predict_roberta_proba(text)
    pred_rob   = int(rob_prob[1] > 0.5)
    pred_bart  = predict_bart(text)
    pred_llama = predict_llama(text)
    trans_vote = int((pred_rob + pred_bart + pred_llama) >= 2)
    trans_conf = float(max(rob_prob))
    timings['Transformers EWC HV'] = time.perf_counter() - t0

    # ── Niveau 1 : GNNs ───────────────────────────────────────
    t1 = time.perf_counter()
    gnn_vote, gnn1_pred, gnn2_pred, gnn3_pred = predict_gnns(text, source=source)
    gnn_conf = 0.92 if gnn_vote == trans_vote else 0.61
    timings['GNNs HV'] = time.perf_counter() - t1

    # ── Niveau 1 : RAG ────────────────────────────────────────
    t2 = time.perf_counter()
    rag_pred, retrieved_docs, rag_conf = predict_rag(text)
    timings['RAG v4'] = time.perf_counter() - t2

    # ── Niveau 2 : Stacking ───────────────────────────────────
    WEIGHTS    = np.array([0.35, 0.30, 0.35])
    votes_arr  = np.array([float(trans_vote), float(gnn_vote), float(rag_pred)])
    meta_score = float(votes_arr @ WEIGHTS)

    if 0.40 <= meta_score <= 0.60:
        x_new      = np.array([[trans_vote, gnn_vote, rag_pred]], dtype=float)
        final_pred = int(meta_lr.predict(x_new)[0])
        final_prob = meta_lr.predict_proba(x_new)[0]
        method     = 'LogReg'
    else:
        final_pred = 1 if meta_score > 0.5 else 0
        fake_p     = min(meta_score + 0.1, 0.999)
        final_prob = np.array([1 - fake_p, fake_p])
        method     = 'Weighted HV'

    # ── SHAP stacking ─────────────────────────────────────────
    x_shap    = np.array([[trans_vote, gnn_vote, rag_pred]], dtype=float)
    shap_s    = explainer_lr.shap_values(x_shap)
    shap_vals = shap_s[1][0] if isinstance(shap_s, list) else shap_s[0]

    # ── SHAP tokens RoBERTa EWC ───────────────────────────────
    top_tokens, token_shap = get_shap_tokens(text, n_top=15)

    total_time = time.perf_counter() - t0

    return {
        'label':         'FAKE' if final_pred == 1 else 'REAL',
        'confidence':    float(max(final_prob)),
        'fake_prob':     float(final_prob[1]),
        'real_prob':     float(final_prob[0]),
        'method':        method,
        'total_time':    total_time,
        'timings':       timings,
        'pred_roberta':  'FAKE' if pred_rob   else 'REAL',
        'pred_bart':     'FAKE' if pred_bart  else 'REAL',
        'pred_llama':    'FAKE' if pred_llama else 'REAL',
        'trans_vote':    trans_vote,
        'trans_conf':    trans_conf,
        'pred_gnn1':     'FAKE' if gnn1_pred else 'REAL',
        'pred_gnn2':     'FAKE' if gnn2_pred else 'REAL',
        'pred_gnn3':     'FAKE' if gnn3_pred else 'REAL',
        'gnn_vote':      gnn_vote,
        'gnn_conf':      gnn_conf,
        'rag_vote':      rag_pred,
        'rag_conf':      rag_conf,
        'retrieved_docs': retrieved_docs[:5],
        'shap_stacking': shap_vals.tolist(),
        'shap_features': ['Transformers EWC', 'GNNs', 'RAG'],
        'top_tokens':    top_tokens,
        'token_shap':    token_shap.tolist(),
    }

print('✅ predict_full() prête')


✅ predict_full() prête


In [8]:
# ================================================================
# CELL 8 — Fonctions de visualisation
# ================================================================

def plot_verdict(result):
    fig, ax = plt.subplots(figsize=(10, 1.6))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    color = '#E24B4A' if result['label'] == 'FAKE' else '#2E7D32'
    icon  = '⚠️' if result['label'] == 'FAKE' else '✅'
    ax.barh(0.5, result['fake_prob'], left=0,                    height=0.55, color='#E24B4A', alpha=0.85)
    ax.barh(0.5, result['real_prob'], left=result['fake_prob'],  height=0.55, color='#2E7D32', alpha=0.85)
    ax.text(-0.01, 0.5, 'REAL', va='center', ha='right', fontsize=9,  color='#2E7D32', fontweight='bold')
    ax.text(1.01,  0.5, 'FAKE', va='center', ha='left',  fontsize=9,  color='#E24B4A', fontweight='bold')
    ax.text(0.5, 0.5,
            f'{icon}  Verdict: {result["label"]}  ({result["confidence"]*100:.1f}% confidence)  [{result["method"]}]',
            va='center', ha='center', fontsize=13, fontweight='bold', color='white')
    plt.tight_layout(); plt.show()

def plot_families_and_shap(result):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ax = axes[0]
    families = ['Transformers EWC\n(RoBERTa+BART+LLaMA)', 'GNNs\n(GCN+GAT+GraphSAGE)', 'RAG v4\n(CRAG)']
    confs    = [result['trans_conf'], result['gnn_conf'], result['rag_conf']]
    votes    = [result['trans_vote'], result['gnn_vote'], result['rag_vote']]
    colors_f = ['#2563EB', '#6D28D9', '#D97706']
    labels_v = ['FAKE' if v == 1 else 'REAL' for v in votes]
    bars = ax.bar(families, [c * 100 for c in confs], color=colors_f,
                  edgecolor='white', width=0.45, alpha=0.88)
    for bar, lbl, v in zip(bars, labels_v, votes):
        clr = '#E24B4A' if v == 1 else '#2E7D32'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                lbl, ha='center', fontsize=11, fontweight='bold', color=clr)
    ax.set_ylim(0, 110); ax.set_ylabel('Confidence (%)', fontsize=10)
    ax.set_title('Level-1 Family Votes (Transformers EWC)', fontsize=12, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax2 = axes[1]
    feats = result['shap_features']
    vals  = result['shap_stacking']
    colors_s = ['#E24B4A' if v > 0 else '#2E7D32' for v in vals]
    ax2.barh(feats, vals, color=colors_s, edgecolor='white', height=0.4, alpha=0.88)
    ax2.axvline(0, color='black', linewidth=0.8, linestyle='--')
    for i, (v, f) in enumerate(zip(vals, feats)):
        ax2.text(v + (0.002 if v >= 0 else -0.002), i,
                 f'{v:+.4f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=10)
    ax2.set_title('SHAP Contribution (Stacking Level-2)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('SHAP value → pushes toward FAKE', fontsize=9)
    ax2.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

def plot_shap_tokens(result):
    tokens = result['top_tokens']
    vals   = result['token_shap']
    if len(tokens) == 0: return
    colors = ['#E24B4A' if v > 0 else '#2E7D32' for v in vals]
    fig, ax = plt.subplots(figsize=(12, max(4, len(tokens) * 0.4)))
    y_pos = range(len(tokens))
    ax.barh(list(y_pos), vals, color=colors, edgecolor='white', height=0.6, alpha=0.88)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(tokens, fontsize=11)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title('SHAP Token Attribution — RoBERTa EWC (FAKE class)', fontsize=13, fontweight='bold')
    ax.set_xlabel('SHAP value → red = pushes toward FAKE, green = pushes toward REAL', fontsize=9)
    fake_patch = mpatches.Patch(color='#E24B4A', label='Pushes → FAKE')
    real_patch = mpatches.Patch(color='#2E7D32', label='Pushes → REAL')
    ax.legend(handles=[fake_patch, real_patch], loc='lower right', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

def plot_rag_docs(result):
    docs = result.get('retrieved_docs', [])
    if not docs:
        print('No retrieved documents.')
        return
    print('\n📚 RAG — Top retrieved documents from Knowledge Base:')
    print('─' * 70)
    for i, d in enumerate(docs, 1):
        lbl_color = '🔴 FAKE' if d['label_int'] == 1 else '🟢 REAL'
        print(f'  [{i}] {lbl_color} | score={d["wscore"]:.3f} | source={d["source"][:40]}')
        print(f'      {d["text"][:100]}...')
        print()

print('✅ Fonctions de visualisation prêtes')


✅ Fonctions de visualisation prêtes


In [9]:
# ================================================================
# CELL 9 — Interface interactive (soutenance)
# ================================================================
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

display(HTML("""
<style>
.demo-header {
    background: linear-gradient(135deg, #1e293b, #0f172a);
    color: white; padding: 16px 22px; border-radius: 12px;
    margin-bottom: 16px; font-family: sans-serif;
}
.demo-header h2 { margin: 0 0 6px; font-size: 20px; }
.demo-header p  { margin: 0; font-size: 13px; opacity: 0.7; }
.demo-tag {
    display: inline-block; background: rgba(255,255,255,0.15);
    border-radius: 6px; padding: 3px 10px; font-size: 11px; margin-right: 6px;
}
</style>
<div class="demo-header">
  <h2>🛡️ Fake News Detection System — Soutenance Demo (EWC)</h2>
  <p>
    <span class="demo-tag">RoBERTa EWC</span>
    <span class="demo-tag">BART EWC</span>
    <span class="demo-tag">LLaMA 3.2-1B EWC</span>
    <span class="demo-tag">GNN (GCN+GAT+GraphSAGE)</span>
    <span class="demo-tag">CRAG v4</span>
    <span class="demo-tag">SHAP XAI</span>
    <span class="demo-tag">Level-2 Stacking</span>
  </p>
</div>
"""))

EXAMPLES = {
    'Conspiracy (FAKE)': (
        "Scientists and doctors are now warning that 5G towers are directly responsible "
        "for weakening the human immune system. Multiple studies suppressed by Big Pharma "
        "show that electromagnetic radiation destroys white blood cells. The government is "
        "covering up the truth to protect telecom industry profits. Natural cures like "
        "colloidal silver have been proven to reverse the damage, but mainstream medicine "
        "refuses to acknowledge this.",
        'unknown'
    ),
    'Political Disinfo (FAKE)': (
        "BREAKING: Leaked documents from inside the White House reveal that the election "
        "results were manipulated by a network of deep state operatives using foreign servers. "
        "Intelligence sources confirm that millions of fraudulent ballots were counted in "
        "swing states while legitimate votes were discarded. The mainstream media is complicit "
        "in hiding this massive fraud from the American people.",
        'unknown'
    ),
    'Reuters (REAL)': (
        "The Federal Reserve raised its benchmark interest rate by a quarter of a percentage "
        "point on Wednesday, pushing borrowing costs to their highest level in 22 years as it "
        "continues its campaign to bring down stubborn inflation. The decision was unanimous "
        "among Federal Open Market Committee members. Fed Chair Jerome Powell said the central "
        "bank remains committed to returning inflation to its 2 percent target.",
        'reuters.com'
    ),
    'NASA Science (REAL)': (
        "NASA scientists confirmed Thursday that the James Webb Space Telescope has captured "
        "the deepest infrared images of the universe ever taken. The images reveal thousands "
        "of galaxies, including some of the faintest objects ever observed, some dating back "
        "more than 13 billion years. The telescope, launched in December 2021, will allow "
        "astronomers to observe galaxy formation in the early universe.",
        'nasa.gov'
    ),
}

example_dd = widgets.Dropdown(
    options=['— paste your own —'] + list(EXAMPLES.keys()),
    description='Example:',
    layout=widgets.Layout(width='50%')
)
text_input = widgets.Textarea(
    placeholder='Paste article text here...',
    layout=widgets.Layout(width='100%', height='130px')
)
source_input = widgets.Text(
    placeholder='Source URL or name (e.g. reuters.com, unknown)',
    description='Source:',
    layout=widgets.Layout(width='55%')
)
btn_analyze = widgets.Button(
    description='🔍 Analyze', button_style='primary',
    layout=widgets.Layout(width='160px', height='38px')
)
btn_clear = widgets.Button(
    description='🗑️ Clear', button_style='warning',
    layout=widgets.Layout(width='110px', height='38px')
)
out = widgets.Output()

def on_example_change(change):
    if change['new'] != '— paste your own —':
        txt, src = EXAMPLES[change['new']]
        text_input.value   = txt
        source_input.value = src

def on_clear(b):
    text_input.value   = ''
    source_input.value = ''
    example_dd.value   = '— paste your own —'
    with out: clear_output()

def on_analyze(b):
    with out:
        clear_output()
        text   = text_input.value.strip()
        source = source_input.value.strip() or 'unknown'
        if not text:
            print('⚠️  Please paste article text first.')
            return
        print(f'🔍 Analyzing... (source: {source})')
        try:
            result = predict_full(text, source=source)
        except Exception as e:
            print(f'❌ Error: {e}')
            import traceback; traceback.print_exc()
            return
        print()
        plot_verdict(result)
        plot_families_and_shap(result)
        print('\n🔑 SHAP Token Attribution (RoBERTa EWC — which words drove the decision):')
        plot_shap_tokens(result)
        plot_rag_docs(result)
        print('─' * 60)
        print(f'  Verdict      : {result["label"]}')
        print(f'  Confidence   : {result["confidence"]*100:.1f}%')
        print(f'  Method       : {result["method"]}')
        print()
        print('  ── Transformers EWC ─────────────────────────')
        print(f'  RoBERTa EWC  : {result["pred_roberta"]}  {"✅" if result["pred_roberta"]  == result["label"] else "❌"}')
        print(f'  BART    EWC  : {result["pred_bart"]}  {"✅" if result["pred_bart"]    == result["label"] else "❌"}')
        print(f'  LLaMA   EWC  : {result["pred_llama"]}  {"✅" if result["pred_llama"]   == result["label"] else "❌"}')
        print(f'  → Trans HV   : {"FAKE" if result["trans_vote"] else "REAL"}')
        print()
        print('  ── GNNs ─────────────────────────────────────')
        print(f'  GNN1 (Source GCN)     : {result["pred_gnn1"]}  {"✅" if result["pred_gnn1"] == result["label"] else "❌"}')
        print(f'  GNN2 (Relational GAT) : {result["pred_gnn2"]}  {"✅" if result["pred_gnn2"] == result["label"] else "❌"}')
        print(f'  GNN3 (Knowledge GNN)  : {result["pred_gnn3"]}  {"✅" if result["pred_gnn3"] == result["label"] else "❌"}')
        print(f'  → GNN HV              : {"FAKE" if result["gnn_vote"] else "REAL"}')
        print()
        print('  ── RAG v4 ───────────────────────────────────')
        print(f'  RAG v4       : {"FAKE" if result["rag_vote"] else "REAL"}')
        print()
        print(f'  Total time   : {result["total_time"]:.2f}s')
        print('─' * 60)

example_dd.observe(on_example_change, names='value')
btn_analyze.on_click(on_analyze)
btn_clear.on_click(on_clear)

display(
    example_dd,
    text_input,
    source_input,
    widgets.HBox([btn_analyze, btn_clear]),
    out
)


Dropdown(description='Example:', layout=Layout(width='50%'), options=('— paste your own —', 'Conspiracy (FAKE)…

Textarea(value='', layout=Layout(height='130px', width='100%'), placeholder='Paste article text here...')

Text(value='', description='Source:', layout=Layout(width='55%'), placeholder='Source URL or name (e.g. reuter…

Output()

In [16]:
import time

article = """
The United States economy added 272,000 jobs in May 2024, surpassing 
economists expectations and demonstrating continued resilience in the 
labor market despite elevated interest rates.
"""
source = "reuters.com"

start = time.time()
result = predict_full(article, source)
end = time.time()

print(f'\n⏱️ Total time: {end-start:.2f}s')


⏱️ Total time: 4.96s


OOD bien predit

FAKE OOD :

Internal documents leaked from Pfizer reveal that the COVID-19 mRNA vaccine 
was designed to permanently alter human DNA and reduce fertility rates by 
30 percent over two generations. The documents, authenticated by three 
independent virologists, show that senior executives knew about these effects 
before the vaccine received emergency use authorization. The World Health 
Organization and Centers for Disease Control deliberately suppressed the 
findings after receiving billions in funding from the Bill and Melinda Gates 
Foundation. Millions of vaccinated individuals are now expected to experience 
chromosomal damage within the next decade.  (bien predit)

unknown

# FAKE 1 — Santé 2024
article1 = """
A new peer-reviewed study published in the Lancet has confirmed that 5G 
radiation causes rapid DNA strand breaks and accelerates tumor growth in 
human tissue. Researchers at MIT found that exposure to 5G frequencies for 
as little as 30 minutes per day increases cancer risk by 47 percent. The 
Federal Communications Commission has been pressured by telecom lobbyists 
to suppress the findings. Several European countries are now quietly 
preparing to dismantle their 5G infrastructure following the revelations.
"""
source1 = "thelancet.com"
label1  = 1  # FAKE

# FAKE 2 — Politique 2024
article2 = """
Leaked State Department cables published by WikiLeaks confirm that Hillary 
Clinton personally ordered the assassination of four American journalists 
who were investigating Clinton Foundation corruption in Haiti. The cables, 
verified by three former intelligence officials, show wire transfers of 
2.3 million dollars to a private security contractor based in Dubai. 
The FBI has opened a criminal investigation but senior officials have been 
ordered by the Attorney General to halt all proceedings until after the 
upcoming midterm elections.
"""
source2 = "wikileaks.org"
label2  = 1  # FAKE

Article 4 (CERN) — REAL 55% ❌ alors que c'est FAKE.
Le problème : le RAG récupère des vrais articles sur le LHC/CERN qui sont très similaires au texte → score REAL élevé. C'est un cas difficile car le contenu scientifique légitime masque la désinformation.
C'est intéressant pour le mémoire — montre que les fake news habillées en langage scientifique sont les plus difficiles à détecter. C'est une limite connue des systèmes RAG.


Scientists at CERN have confirmed that the Large Hadron Collider experiment 
conducted in September 2024 successfully opened a microscopic dimensional 
portal that allowed communication with a parallel universe for 0.3 seconds. 
The breakthrough, classified as top secret by twelve governments, proves 
the existence of the multiverse and has profound implications for quantum 
physics. Three researchers who witnessed the event have since disappeared 
under mysterious circumstances, raising concerns about a government coverup. (cern.ch)

article = """
Newly released FBI records obtained by Judicial Watch reveal that Deputy 
Kinlee Hoyle had two email exchanges with Thomas Matthew Crooks prior to 
the July 13, 2024 assassination attempt on President Trump in Butler, 
Pennsylvania. The emails content remains heavily redacted, but the deputy 
confirmed the communications in post-incident interviews. This adds another 
layer of questions about pre-rally interactions between law enforcement 
and the shooter.
"""
source = "leadstories.com"
label  = 1  # FAKE
(bien predit)

article = """
RIP Jordan Peterson (1982-2026)
"""
source = "unknown"
label  = 1  # FAKE
bien predit

REAL bien predits 


# REAL 1 — UEFA Euro 2024
article1 = """
Spain won the UEFA European Championship 2024, defeating England 2-1 in the 
final held in Berlin on July 14, 2024. Mikel Oyarzabal scored the winning 
goal in the 86th minute after Cole Palmer had equalised for England. Spain 
became the first nation to win four European Championships. Manager Luis 
de la Fuente praised his squad's resilience throughout the tournament. 
England manager Gareth Southgate resigned following the defeat, ending 
his eight-year tenure as national team manager.
"""
source1 = "uefa.com"
label1  = 0  # REAL